# 04 - T1/T2 Relaxation Fitting *(new in v0.4.0)*

`epyr.relaxation` fits time-domain decay and recovery curves. It mirrors the
lineshape fitting API:

- `fit_relaxation(t, y, model=...)` -> `RelaxationFitResult`
- `fit_multiple_decays(t, y)` -> `RelaxationFitComparison` (a dict that prints as a table)

Models: `mono_exponential`, `stretched_exponential`, `biexponential`,
`inversion_recovery`, `saturation_recovery`, `gamma_gaussian_decay`.

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings("ignore")  # keep tutorial output readable

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

import epyr

DATA = Path("..") / "data"   # example datasets, relative to this notebook
print("EPyR Tools version:", epyr.__version__)

In [ ]:
from epyr.relaxation import (
    fit_relaxation, fit_multiple_decays,
    biexponential, gamma_gaussian_decay,
)

## Real T2 Hahn-echo decay

Echo-detected data is complex; take the magnitude before fitting. The decay is
nearly mono-exponential but often fits better as a stretched exponential.

In [ ]:
t, y, params, _ = epyr.eprload(DATA / "2020_10_DMTTFBr_T2EH_28dB_6K_20ns_40ns_hperpc.DTA",
                               plot_if_possible=False)
y_mag = np.abs(y)

result = fit_relaxation(t, y_mag, model="stretched_exponential", time_unit="ns", plot=True)
plt.show()
print(result)   # RelaxationFitResult prints its own summary

## Comparing decay models

`fit_multiple_decays` ranks candidate models by reduced chi^2 (not R^2, which is
biased toward models with more free parameters). The returned object prints as a
side-by-side table.

In [ ]:
comparison = fit_multiple_decays(
    t, y_mag, models=["mono_exponential", "stretched_exponential"], plot=True)
plt.show()
print(comparison)

## Synthetic bi-exponential decay

Two components with well-separated time constants.

In [ ]:
tt = np.linspace(0, 20, 300)
yy = biexponential(tt, amplitude1=1.0, tau1=1.5, amplitude2=0.6, tau2=8.0, offset=0.05)
yy = yy + np.random.normal(0, 0.01, tt.size)

res_bi = fit_relaxation(tt, yy, model="biexponential", plot=True)
plt.show()
print(res_bi.summary())

## Combined homogeneous / spectral-diffusion decay

`gamma_gaussian_decay` models an echo decay with an exponential (homogeneous,
`Gamma0`) and a Gaussian (spectral diffusion, `GammaG`) contribution.

In [ ]:
tg = np.linspace(0, 5, 300)
yg = gamma_gaussian_decay(tg, amplitude=2.0, Gamma0=0.3, GammaG=0.5, offset=0.1)
yg = yg + np.random.normal(0, 0.01, tg.size)

res_g = fit_relaxation(tg, yg, model="gamma_gaussian_decay", time_unit="us", plot=True)
plt.show()
print(res_g.summary())

## Summary

- Take `np.abs` of complex echo data before fitting.
- `fit_relaxation` fits one model; `fit_multiple_decays` compares and ranks by reduced chi^2.
- Six models cover mono/stretched/bi-exponential decay, recovery, and combined decay.